# House Stage 1: why is 2026 red? Incumbency, midterm and generic-ballot variants (2026-11-03)

**Question.** The 2026 House forecast is redder than Kalshi: in the D-vs-R races, the model expects about 15 fewer
Democratic seats than the market, and almost all of the gap is in Republican-held seats. Is the incumbency term the
reason, or is it the generic ballot or a missing midterm effect?

Production Model A is `adj_margin_2p_norm ~ partisan_lean + gen_margin_raw + 2.5·inc_signed + partisan_lean:inc_signed`
(`src/house/stage1.py`). Here its incumbency treatment is changed and everything else (Model B, equal-year WLS, the
windows, the H0 error model, race rules) is left exactly as in production:

| treatment | Model A incumbency terms |
|---|---|
| `inc_2.5` (production) | `2.5·inc_signed` fixed + `partisan_lean:inc_signed` estimated |
| `no_inc` | none: `inc_signed` and the interaction removed |
| `inc_1_flat` | `1.0·inc_signed` fixed, no interaction |
| `inc_1_interact` | `1.0·inc_signed` fixed + the interaction estimated (separates the two terms' effects) |
| `midterm_pres_inc` | production + `pres_inc_midterm` estimated: `inc_signed` for an incumbent of the president's party in a midterm, else 0 (a negative coefficient means those incumbents lose part of their edge) |
| `gb_1.0` | production with `gen_margin_raw`'s coefficient fixed at 1.0 (the generic ballot passed through in full) |

Each treatment is run as Model A alone and as the 85/15 A/B ensemble (Model B unchanged; it has no incumbency
term). The comparisons are:
1. Stage 1 rolling out-of-sample accuracy, 2018-2024 (each year fit on 2006 through the election before it)
2. Stage 2 (H0) backtest, 2020-2024: Brier and coverage; and the 2018 test (the only out-of-sample midterm under a
   Republican president, as in 2026), with Stage 2 fit on 2012-2016 as in `02_monte_carlo.ipynb`
3. The 2026 forecast: seats, and the gap to Kalshi by incumbent party (Kalshi pull `2026-11-03T0912Z`, read from
   `data/processed/kalshi/kalshi_odds_log_2026.csv`; no API call)

Nothing here writes a production file.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr
from house import simulation, stage1

pd.set_option("display.width", 200)
with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()
    targets = hr.load_targets()
targets_raw = simulation.load_targets_raw()
# pres_inc_midterm: inc_signed for an incumbent of the president's party in a midterm year, else 0
predictors = stage1.add_derived_predictors(predictors)
predictors["pres_inc_midterm"] = np.where((predictors["is_midterm"] == 1)
                                          & (predictors["inc_signed"] == predictors["pres_party_signed"]),
                                          predictors["inc_signed"], 0)
candidates = simulation.load_candidates_2026()

## Settings and variants

In [2]:
TARGET = stage1.TARGET
FORECAST_YEAR = 2026
HOUSE_SEATS = 435
CALIBRATION_YEARS = [2020, 2022, 2024]  # as in 02_monte_carlo.ipynb; 2018 only initializes the 2020 fit
SEED = 42
N_SIMS = 40_000
NU = 5
NATIONAL_PRIOR_SD = 3.0
NATIONAL_PRIOR_ELECTIONS = 2
KALSHI_LOG_PATH = ROOT / "data/processed/kalshi/kalshi_odds_log_2026.csv"
KALSHI_PULL = "2026-11-03T0912Z"

# Model A's incumbency treatments (everything else in the formula is production's)
A_COMPONENTS = {
    "inc_2.5": stage1.MODEL_A,  # production
    "no_inc": stage1.Component(
        "A_no_inc",
        f"{TARGET} ~ partisan_lean + gen_margin_raw",
    ),
    "inc_1_flat": stage1.Component(
        "A_inc_1_flat",
        f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed",
        fixed={"inc_signed": 1.0},
    ),
    "inc_1_interact": stage1.Component(
        "A_inc_1_interact",
        f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed",
        fixed={"inc_signed": 1.0},
    ),
    "midterm_pres_inc": stage1.Component(
        "A_midterm_pres_inc",
        f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + pres_inc_midterm",
        fixed={"inc_signed": 2.5},
    ),
    "gb_1.0": stage1.Component(
        "A_gb_1",
        f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed",
        fixed={"inc_signed": 2.5, "gen_margin_raw": 1.0},
    ),
}
# (treatment, model) -> config; the ensemble's fallback is the same treatment's Model A, as in production
CONFIGS = {}
for treatment, component in A_COMPONENTS.items():
    CONFIGS[(treatment, "A")] = stage1.Stage1Config(f"A_{treatment}", ((component, 1.0),))
    CONFIGS[(treatment, "AB_85_15")] = stage1.Stage1Config(f"AB_85_15_{treatment}",
                                                           ((component, 0.85), (stage1.MODEL_B, 0.15)))
pd.Series({key: " + ".join(f"{w:g}·{c.formula.split('~')[1].strip()} {dict(c.fixed) or ''}"
                           for c, w in config.components) for key, config in CONFIGS.items()}, name="spec").to_frame()

spec
inc_2.5          A         1·partisan_lean + gen_margin_raw + inc_signed ...
                 AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
no_inc           A                         1·partisan_lean + gen_margin_raw 
                 AB_85_15  0.85·partisan_lean + gen_margin_raw  + 0.15·pa...
inc_1_flat       A         1·partisan_lean + gen_margin_raw + inc_signed ...
                 AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
inc_1_interact   A         1·partisan_lean + gen_margin_raw + inc_signed ...
                 AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
midterm_pres_inc A         1·partisan_lean + gen_margin_raw + inc_signed ...
                 AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
gb_1.0           A         1·partisan_lean + gen_margin_raw + inc_signed ...
                 AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...

## 1. Stage 1 rolling out-of-sample accuracy, 2018-2024

Bias is actual − predicted, in points of Democratic margin: positive means the model was too Republican.

In [3]:
runs = {key: simulation.stage1_inputs(config, predictors=predictors, targets=targets) for key, config in CONFIGS.items()}

def pooled_metrics(run):
    rows = {}
    for subset in ["valid", "competitive_ex_ante"]:
        scored = run.oos if subset == "valid" else run.oos[run.oos["competitive_ex_ante"]]
        rows[(subset, "MAE")] = scored["residual"].abs().mean()
        rows[(subset, "bias")] = scored["residual"].mean()
        rows[(subset, "called %")] = 100 * (np.sign(scored["prediction"]) == np.sign(scored["actual"])).mean()
        rows[(subset, "n")] = len(scored)
    return pd.Series(rows)

stage1_table = pd.DataFrame({key: pooled_metrics(run) for key, run in runs.items()}).T
stage1_table.index.names = ["treatment", "model"]
stage1_table.round(2)

valid                        competitive_ex_ante                      
                            MAE  bias called %       n                 MAE  bias called %      n
treatment        model                                                                          
inc_2.5          A         5.48 -0.28    96.12  1598.0                4.99 -0.30    86.12  353.0
                 AB_85_15  5.17 -0.47    96.12  1598.0                4.68 -0.53    86.12  353.0
no_inc           A         5.68 -0.61    94.87  1598.0                5.67  1.09    80.45  353.0
                 AB_85_15  5.28 -0.75    95.12  1598.0                5.22  0.65    81.59  353.0
inc_1_flat       A         5.54 -0.65    95.06  1598.0                5.38  1.06    81.30  353.0
                 AB_85_15  5.17 -0.78    95.49  1598.0                4.97  0.62    83.29  353.0
inc_1_interact   A         5.60 -0.18    95.87  1598.0                5.13 -0.44    84.99  353.0
                 AB_85_15  5.25 -0.38    96.06  1598.0                4.81 -0.65    85.84  353.0
midterm_pres_inc A         6.40  0.20    95.62  1598.0                6.75  1.17    84.42  353.0
                 AB_85_15  5.84 -0.06    95.62  1598.0                5.98  0.72    84.14  353.0
gb_1.0           A         5.58 -0.24    96.18  1598.0                4.91 -0.36    86.40  353.0
                 AB_85_15  5.28 -0.43    96.12  1598.0                4.65 -0.58    86.12  353.0

In [4]:
# Bias by year (actual − predicted, all valid districts): is the sign stable, or does it flip with the environment?
bias_by_year = pd.DataFrame({key: run.oos.groupby("year")["residual"].mean() for key, run in runs.items()}).T
bias_by_year.index.names = ["treatment", "model"]
bias_by_year.round(2)

year                       2018  2020  2022  2024
treatment        model                           
inc_2.5          A         2.37 -3.15  0.83 -1.08
                 AB_85_15  1.23 -3.04  1.49 -1.46
no_inc           A         1.69 -3.10  0.49 -1.45
                 AB_85_15  0.65 -3.00  1.20 -1.77
inc_1_flat       A         1.71 -3.28  0.49 -1.41
                 AB_85_15  0.66 -3.16  1.20 -1.74
inc_1_interact   A         2.44 -2.83  0.87 -1.10
                 AB_85_15  1.28 -2.77  1.53 -1.48
midterm_pres_inc A         5.89 -4.83 -0.41  0.34
                 AB_85_15  4.22 -4.47  0.44 -0.26
gb_1.0           A         1.27 -3.73  2.00 -0.39
                 AB_85_15  0.29 -3.54  2.49 -0.88

In [5]:
# The same bias in ex-ante competitive districts, split by who held the seat (inc_signed: +1 D, -1 R, 0 open)
inc = predictors[[*hr.KEY, "inc_signed"]]
by_holder = {}
for key, run in runs.items():
    d = run.oos[run.oos["competitive_ex_ante"]].merge(inc, on=hr.KEY)
    by_holder[key] = d.groupby(["inc_signed", "year"])["residual"].mean().unstack("year")
pd.concat(by_holder, names=["treatment", "model"]).round(2)

year                                   2018  2020  2022  2024
treatment        model    inc_signed                         
inc_2.5          A        -1           2.70 -3.91 -5.07 -3.49
                           0           3.09 -0.50 -1.35 -0.35
                           1           5.29 -1.35  1.33  0.17
                 AB_85_15 -1           1.59 -3.85 -4.09 -3.72
                           0           1.95 -0.94 -0.54 -0.76
                           1           4.44 -1.19  1.91 -0.46
no_inc           A        -1           1.44 -5.20 -5.82 -4.44
                           0           5.62  1.93  0.43  1.27
                           1          10.53  3.82  5.24  3.99
                 AB_85_15 -1           0.51 -4.95 -4.72 -4.53
                           0           4.11  1.13  0.98  0.61
                           1           8.89  3.21  5.23  2.79
inc_1_flat       A        -1           2.10 -4.56 -4.84 -3.52
                           0           5.31  1.62  0.50  1.34
                           1           9.38  2.65  4.33  3.07
                 AB_85_15 -1           1.08 -4.41 -3.89 -3.74
                           0           3.84  0.86  1.03  0.68
                           1           7.91  2.22  4.46  2.00
inc_1_interact   A        -1           1.65 -4.92 -6.77 -5.07
                           0           3.32 -0.25 -1.71 -0.73
                           1           6.70  0.11  2.47  1.33
                 AB_85_15 -1           0.70 -4.71 -5.53 -5.06
                           0           2.15 -0.72 -0.85 -1.08
                           1           5.64  0.06  2.88  0.53
midterm_pres_inc A        -1          12.16 -6.11 -2.52 -2.00
                           0           0.46 -2.17  1.42  1.51
                           1           3.30 -2.48 -3.92  1.99
                 AB_85_15 -1           9.63 -5.72 -1.92 -2.45
                           0          -0.28 -2.36  1.82  0.82
                           1           2.75 -2.15 -2.56  1.08
gb_1.0           A        -1           1.62 -4.48 -3.95 -2.82
                           0           2.00 -1.10 -0.24  0.30
                           1           4.19 -1.96  2.45  0.83
                 AB_85_15 -1           0.67 -4.34 -3.13 -3.15
                           0           1.03 -1.44  0.40 -0.21
                           1           3.51 -1.71  2.86  0.10

## 2. Stage 2 (H0) backtest, 2020-2024

In [6]:
races = simulation.historical_races(targets, targets_raw)
calibration = {}
for (treatment, model), run in runs.items():
    calibration[(treatment, model)] = simulation.backtest(
        run.oos,
        test_years=CALIBRATION_YEARS,
        races=races,
        predictions=run.predictions,
        fallback_predictions=runs[(treatment, "A")].predictions,
        nu=NU,
        n_sims=N_SIMS,
        seed=SEED,
        house_size=HOUSE_SEATS,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
    )
stage2_table = pd.DataFrame({key: cal.pooled() for key, cal in calibration.items()}).T
stage2_table.index.names = ["treatment", "model"]
stage2_table[["Brier (all races)", "contested Brier", "competitive Brier", "80% coverage", "95% coverage"]].round(4)

Brier (all races)  contested Brier  competitive Brier  80% coverage  95% coverage
treatment        model                                                                                      
inc_2.5          A                    0.0271           0.0293             0.1020        0.8515        0.9701
                 AB_85_15             0.0266           0.0288             0.0995        0.8490        0.9643
no_inc           A                    0.0353           0.0382             0.1423        0.8456        0.9610
                 AB_85_15             0.0334           0.0361             0.1328        0.8407        0.9593
inc_1_flat       A                    0.0321           0.0348             0.1267        0.8473        0.9627
                 AB_85_15             0.0308           0.0333             0.1199        0.8407        0.9585
inc_1_interact   A                    0.0301           0.0326             0.1164        0.8523        0.9701
                 AB_85_15             0.0291           0.0315             0.1115        0.8498        0.9660
midterm_pres_inc A                    0.0277           0.0300             0.1068        0.9178        0.9859
                 AB_85_15             0.0265           0.0287             0.1013        0.8963        0.9834
gb_1.0           A                    0.0276           0.0298             0.1037        0.8490        0.9693
                 AB_85_15             0.0271           0.0293             0.1015        0.8382        0.9627

In [7]:
# Seats: median simulated vs actual Democratic seats per test year
seats = pd.concat({key: cal.seats() for key, cal in calibration.items()}, names=["treatment", "model"])
seats

matched panel unresolved  median panel_dem_seats  panel_actual_dem_seats  median − actual  actual-seat percentile  called D seats  called − actual
treatment        model    test year                                                                                                                                                   
inc_2.5          A        2020         435 of 435                                231                     222                9                15.22375             234               12
                          2022         435 of 435                                212                     213               -1                54.45000             213                0
                          2024         435 of 435                                215                     215                0                52.25500             214               -1
                 AB_85_15 2020         435 of 435                                231                     222                9                13.94125             234               12
                          2022         435 of 435                                210                     213               -3                64.12250             211               -2
                          2024         435 of 435                                216                     215                1                45.36250             215                0
no_inc           A        2020         435 of 435                                221                     222               -1                52.42875             221               -1
                          2022         435 of 435                                204                     213               -9                82.22000             207               -6
                          2024         435 of 435                                209                     215               -6                73.76000             211               -4
                 AB_85_15 2020         435 of 435                                222                     222                0                48.46250             221               -1
                          2022         435 of 435                                203                     213              -10                86.62500             207               -6
                          2024         435 of 435                                212                     215               -3                65.74750             212               -3
inc_1_flat       A        2020         435 of 435                                223                     222                1                45.31875             223                1
                          2022         435 of 435                                205                     213               -8                80.45375             208               -5
                          2024         435 of 435                                210                     215               -5                73.84500             210               -5
                 AB_85_15 2020         435 of 435                                224                     222                2                41.40375             225                3
                          2022         435 of 435                                203                     213              -10                85.11875             207               -6
                          2024         435 of 435                                212                     215               -3                65.83625             213               -2
inc_1_interact   A        2020         435 of 435                                230                     222                8                21.60000             231                9
                          2022         435 of 435                                212                     213               -1                55.80375             212               -1
          

In [8]:
# The 2018 test: Stage 1 out-of-sample 2012-2018, Stage 2 fit on 2012-2016 residuals only, scored on 2018
EARLY_TEST_YEAR = 2018
EARLY_STAGE1_EVAL_YEARS = [2012, 2014, 2016, 2018]
EARLY_STAGE2_TRAIN_YEARS = [2012, 2014, 2016]
early_runs = {key: simulation.stage1_inputs(config, predictors=predictors, targets=targets,
                                            eval_years=EARLY_STAGE1_EVAL_YEARS) for key, config in CONFIGS.items()}
early = {}
for (treatment, model), run in early_runs.items():
    early[(treatment, model)] = simulation.backtest(
        run.oos,
        test_years=[EARLY_TEST_YEAR],
        train_years=EARLY_STAGE2_TRAIN_YEARS,
        races=races,
        predictions=run.predictions,
        fallback_predictions=early_runs[(treatment, "A")].predictions,
        nu=NU,
        n_sims=N_SIMS,
        seed=SEED,
        house_size=HOUSE_SEATS,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
    )
early_table = pd.DataFrame({key: cal.pooled() for key, cal in early.items()}).T
early_table.index.names = ["treatment", "model"]
early_seats = pd.concat({key: cal.seats() for key, cal in early.items()}, names=["treatment", "model"])
early_table[["Brier (all races)", "competitive Brier", "80% coverage"]].join(
    early_seats.droplevel("test year")[["median panel_dem_seats", "panel_actual_dem_seats"]]).round(4)

Brier (all races)  competitive Brier  80% coverage  median panel_dem_seats  panel_actual_dem_seats
treatment        model                                                                                                       
inc_2.5          A                    0.0374             0.1327        0.9033                     225                     235
                 AB_85_15             0.0352             0.1249        0.9237                     229                     235
no_inc           A                    0.0412             0.1451        0.9135                     224                     235
                 AB_85_15             0.0377             0.1338        0.9160                     228                     235
inc_1_flat       A                    0.0405             0.1436        0.9109                     223                     235
                 AB_85_15             0.0373             0.1322        0.9135                     227                     235
inc_1_interact   A                    0.0374             0.1319        0.9135                     227                     235
                 AB_85_15             0.0354             0.1249        0.9186                     231                     235
midterm_pres_inc A                    0.0513             0.1856        0.7684                     212                     235
                 AB_85_15             0.0472             0.1720        0.8270                     216                     235
gb_1.0           A                    0.0361             0.1277        0.9109                     229                     235
                 AB_85_15             0.0345             0.1220        0.9135                     232                     235

## 3. The 2026 forecast, and the gap to Kalshi

In [9]:
# Same-party top-two races, exactly as in 02_monte_carlo.ipynb
TOP_TWO_WEIGHTS_WITH_POLL = {"primary": 0.6, "fundraising": 0.3, "poll": 0.1}
TOP_TWO_WEIGHTS_NO_POLL = {"primary": 0.6, "fundraising": 0.4}
TOP_TWO_POINTS_PER_LOG = 4.46
TOP_TWO_BASE_SD = 8.0
TOP_TWO_SPLIT_SD = 0.15
top_two = simulation.predict_top_two(simulation.load_top_two(), weights_with_poll=TOP_TWO_WEIGHTS_WITH_POLL,
                                     weights_no_poll=TOP_TWO_WEIGHTS_NO_POLL, points_per_log=TOP_TWO_POINTS_PER_LOG,
                                     base_sd=TOP_TWO_BASE_SD, split_sd=TOP_TWO_SPLIT_SD, nu=NU)
RESOLVERS = [simulation.TopTwoSameParty(top_two)]

forecasts, draws_2026, tables_2026 = {}, {}, {}
for (treatment, model), config in CONFIGS.items():
    run = runs[(treatment, model)]
    error_sd, _ = simulation.estimate_error_sds(run.oos, national_prior_sd=NATIONAL_PRIOR_SD,
                                                national_prior_elections=NATIONAL_PRIOR_ELECTIONS)
    scale = {level: simulation.t_scale(sd, NU) for level, sd in error_sd.items()}
    forecasts[(treatment, model)] = simulation.stage1_forecast(config, predictors=predictors, targets=targets)
    fallback = simulation.stage1_forecast(CONFIGS[(treatment, "A")], predictors=predictors, targets=targets)
    point = simulation.attach_predictions(simulation.races_2026(candidates), forecasts[(treatment, model)].predictions,
                                          primary_source=config.name, fallback=fallback.predictions)
    draws = simulation.simulate_races(point, scale, NU, N_SIMS, np.random.default_rng([SEED, FORECAST_YEAR]),
                                      resolvers=RESOLVERS)
    draws_2026[(treatment, model)] = draws
    tables_2026[(treatment, model)] = simulation.forecast_table(draws, candidates, resolvers=RESOLVERS)

dem_seats = {key: d.seats(simulation.DEM) for key, d in draws_2026.items()}
seats_2026 = pd.DataFrame({key: {"mean D seats": s.mean(), "median": np.median(s),
                                 "80% interval": f"{int(np.quantile(s, 0.1))}-{int(np.quantile(s, 0.9))}",
                                 "P(D 218+)": (s >= 218).mean()} for key, s in dem_seats.items()}).T
seats_2026.index.names = ["treatment", "model"]
seats_2026

mean D seats median 80% interval P(D 218+)
treatment        model                                              
inc_2.5          A          231.423125  231.0      220-244  0.938725
                 AB_85_15    231.45545  231.0      220-243  0.946275
no_inc           A          228.322325  228.0      216-241   0.86385
                 AB_85_15     228.8982  228.0      217-241   0.88825
inc_1_flat       A            227.7165  227.0      216-240  0.859225
                 AB_85_15     228.3501  228.0      217-241  0.884075
inc_1_interact   A            232.9628  232.0      221-246  0.948425
                 AB_85_15    232.81685  232.0      221-245  0.955025
midterm_pres_inc A             226.199  225.0      213-240    0.8002
                 AB_85_15   226.424725  226.0      215-239   0.83555
gb_1.0           A            235.7485  235.0      223-249  0.969525
                 AB_85_15   235.108275  234.0      223-248  0.968075

In [10]:
# Model A's 2026 coefficients under each treatment (2006-2024 fit): does removing incumbency move the others?
pd.DataFrame({treatment: forecasts[(treatment, "A")].coefficients.iloc[:, 0] for treatment in A_COMPONENTS}).round(3)

,inc_2.5,no_inc,inc_1_flat,inc_1_interact,midterm_pres_inc,gb_1.0
term,,,,,,
Intercept,1.199,-0.492,-0.495,1.451,-0.620,0.466
gen_margin_raw,0.781,0.765,0.771,0.774,1.148,1.000
inc_signed,2.500,NaN,1.000,1.000,2.500,2.500
partisan_lean,1.023,1.063,1.040,1.059,0.980,1.025
partisan_lean:inc_signed,-0.086,NaN,NaN,-0.099,-0.071,-0.087
pres_inc_midterm,NaN,NaN,NaN,NaN,7.076,NaN


In [11]:
# Kalshi's probability that the Democrat wins, for races Kalshi lists as one Democrat vs others (D-vs-R and three-way)
log = pd.read_csv(KALSHI_LOG_PATH).query("pulled_at == @KALSHI_PULL")
dem_markets = log[log["party"] == "D"].groupby("race_id")["kalshi_prob"].agg(["sum", "size"])
kalshi_dem = dem_markets.loc[dem_markets["size"] == 1, "sum"].rename("kalshi_dem")

def race_id(frame):
    return frame["state"] + "-" + frame["district"].map(lambda d: "AL" if d == 0 else f"{d:02d}")

holders = predictors.query("year == @FORECAST_YEAR").assign(race_id=race_id)[["race_id", "inc_signed"]]
comparison = {}
for key, table in tables_2026.items():
    d = (table.query("treatment == 'simulated'").assign(race_id=race_id)[["race_id", "p_dem"]]
         .merge(kalshi_dem, on="race_id").merge(holders, on="race_id"))
    comparison[key] = d.groupby("inc_signed")[["p_dem", "kalshi_dem"]].sum().stack()
    comparison[key]["all", "p_dem"] = d["p_dem"].sum()
    comparison[key]["all", "kalshi_dem"] = d["kalshi_dem"].sum()
gap = pd.DataFrame(comparison).T
gap.index.names = ["treatment", "model"]
gap.columns = [f"{'R-held' if h == -1 else 'open' if h == 0 else 'D-held' if h == 1 else 'all'}: "
               f"{'model' if c == 'p_dem' else 'Kalshi'}" for h, c in gap.columns]
display(Markdown("Expected Democratic seats among the simulated races Kalshi prices (sum of P(D win)), by holder"))
gap.round(1)

Expected Democratic seats among the simulated races Kalshi prices (sum of P(D win)), by holder

R-held: model  R-held: Kalshi  open: model  open: Kalshi  D-held: model  D-held: Kalshi  all: model  all: Kalshi
treatment        model                                                                                                                     
inc_2.5          A                  18.5            32.2         28.5          31.0          167.5           166.7       214.4        229.9
                 AB_85_15           18.4            32.2         28.2          31.0          167.8           166.7       214.5        229.9
no_inc           A                  21.3            32.2         26.8          31.0          163.2           166.7       211.3        229.9
                 AB_85_15           20.7            32.2         26.8          31.0          164.4           166.7       211.9        229.9
inc_1_flat       A                  19.4            32.2         26.9          31.0          164.4           166.7       210.7        229.9
                 AB_85_15           19.1            32.2         27.0          31.0          165.3           166.7       211.4        229.9
inc_1_interact   A                  21.3            32.2         28.3          31.0          166.3           166.7       216.0        229.9
                 AB_85_15           20.8            32.2         28.1          31.0          166.8           166.7       215.8        229.9
midterm_pres_inc A                  10.7            32.2         30.6          31.0          167.9           166.7       209.2        229.9
                 AB_85_15           11.3            32.2         29.9          31.0          168.3           166.7       209.4        229.9
gb_1.0           A                  21.0            32.2         29.5          31.0          168.2           166.7       218.8        229.9
                 AB_85_15           20.6            32.2         29.1          31.0          168.4           166.7       218.1        229.9

In [12]:
# Model A's 2026 point margins under each treatment in the races most out of line with Kalshi
WATCH = ["TX-15", "OH-07", "IA-01", "NC-11", "PA-08", "CO-03", "NY-17", "PA-01", "CA-22", "CO-08"]
margins = pd.DataFrame({t: forecasts[(t, "AB_85_15")].predictions.assign(race_id=race_id).set_index("race_id")
                        ["prediction"] for t in A_COMPONENTS})
p_dem = pd.DataFrame({t: tables_2026[(t, "AB_85_15")].assign(race_id=race_id).set_index("race_id")["p_dem"]
                      for t in A_COMPONENTS})
watch = pd.concat([margins.add_prefix("margin D+ "), p_dem.add_prefix("P(D) ")], axis=1).loc[WATCH]
watch["P(D) Kalshi"] = kalshi_dem.reindex(WATCH)
watch["inc_signed"] = holders.set_index("race_id")["inc_signed"].reindex(WATCH)
watch.round(2)

,margin D+ inc_2.5,margin D+ no_inc,margin D+ inc_1_flat,margin D+ inc_1_interact,margin D+ midterm_pres_inc,margin D+ gb_1.0,P(D) inc_2.5,P(D) no_inc,P(D) inc_1_flat,P(D) inc_1_interact,P(D) midterm_pres_inc,P(D) gb_1.0,P(D) Kalshi,inc_signed
race_id,,,,,,,,,,,,,,
TX-15,-9.11,-7.99,-8.53,-8.25,-13.27,-8.14,0.08,0.11,0.10,0.10,0.04,0.10,0.90,-1
OH-07,-4.82,-3.86,-4.48,-3.80,-9.18,-3.85,0.22,0.27,0.24,0.27,0.09,0.27,0.86,-1
IA-01,-1.90,-1.05,-1.72,-0.76,-6.40,-0.92,0.38,0.44,0.39,0.45,0.18,0.44,0.90,-1
NC-11,-3.13,-2.17,-2.79,-2.11,-7.49,-2.15,0.31,0.37,0.33,0.37,0.14,0.37,0.87,-1
PA-08,-1.76,-0.91,-1.59,-0.62,-6.26,-0.78,0.39,0.44,0.40,0.46,0.18,0.45,0.88,-1
CO-03,-4.93,-3.99,-4.62,-3.88,-9.32,-3.95,0.21,0.26,0.23,0.27,0.09,0.26,0.59,-1
NY-17,8.50,8.96,8.10,10.05,3.52,9.50,0.91,0.91,0.90,0.93,0.70,0.93,0.56,-1
PA-01,6.39,6.91,6.07,7.88,1.47,7.39,0.84,0.85,0.83,0.88,0.59,0.87,0.62,-1
CA-22,7.58,8.07,7.22,9.10,2.63,8.58,0.88,0.89,0.87,0.91,0.65,0.90,0.92,-1


### How far is the production forecast from Kalshi, in points?

Add a uniform shift to every simulated 2026 margin of the production configuration (inc_2.5, AB_85_15) and recount
expected Democratic seats among the races Kalshi prices, overall and in Republican-held seats. A diagnostic only:
no forecast is changed.

In [13]:
SHIFTS = [0, 1, 2, 3, 4, 5, 6]  # points of Democratic margin added to every simulated race
draws = draws_2026[("inc_2.5", "AB_85_15")]
race_ids = race_id(draws.races).to_numpy()
priced = np.isin(race_ids, kalshi_dem.index) & (draws.races["treatment"] == "simulated").to_numpy()
r_held = np.isin(race_ids, holders.loc[holders["inc_signed"] == -1, "race_id"])
rows = {}
for shift in SHIFTS:
    p = ((draws.margins[:, priced] + shift) > 0).mean(axis=0)
    rows[shift] = {"all: model": p.sum(), "R-held: model": p[r_held[priced]].sum()}
shifted = pd.DataFrame(rows).T.rename_axis("shift (D+ points)")
shifted["all: Kalshi"] = kalshi_dem.reindex(race_ids[priced]).sum()
shifted["R-held: Kalshi"] = kalshi_dem.reindex(race_ids[priced & r_held]).sum()
shifted.round(1)

,all: model,R-held: model,all: Kalshi,R-held: Kalshi
shift (D+ points),,,,
0,214.5,18.4,229.9,32.2
1,218.1,20.6,229.9,32.2
2,222.0,22.9,229.9,32.2
3,225.9,25.4,229.9,32.2
4,230.1,28.1,229.9,32.2
5,234.5,31.0,229.9,32.2
6,239.1,34.1,229.9,32.2


## Result (2026-11-03)

Production Model A is kept unchanged.

- **Incumbency** (`no_inc`, `inc_1_flat`, `inc_1_interact`): every variant is worse on Brier and competitive MAE, and
  none is bluer in 2026. Without the term, the intercept falls by about 1.7 points and absorbs most of the change.
- **`midterm_pres_inc`**: the coefficient is +7, so incumbents of the president's party held up *better* in midterms.
  The 2018 test gets much worse (Brier 0.047 vs 0.035), and 2026 gets redder. Rejected.
- **`gb_1.0`**: the backtest is about tied with production and 2026 is about 1 point bluer, but it was rejected:
  imposing the generic ballot's pass-through instead of estimating it isn't justified.
- The 2026 gap to Kalshi (about 4 points of uniform shift, most of it in Republican-held seats) isn't explained by any
  specification tested here. It is left to race-level judgment (`data/reference/track_overrides_2026.csv`).